# Supervised fine-tuning

answer_only takes train_input and eval_input lists, pointing to notebook 01's train_prepared.json and dev_prepared.json. model_name can be Qwen/Qwen2.5-0.5B-Instruct, Qwen/Qwen2.5-3B-Instruct or a local model directory. Other supported models require their matching chat template.

The current split-first data lives in data/BioASQ_factoid_sft_prepared/split_first_train90_dev10_supported_train_seed3407/: 1,137 supported training questions (1,363 rows) and 160 unfiltered dev questions. The earlier 126-dev experiment remains separate. Previously trained adapters used different split membership, so train fresh adapters for the new protocol; their old dev scores are not directly comparable. Recreate plans after changing data paths. Any retained outputs below may describe earlier runs.

Defaults: 4-bit LoRA, rank/alpha 32, dropout 0.05, batch size 1, accumulation 32, up to 6 epochs and generated-dev MRR checkpoint selection. These are inherited experiment settings, not newly tuned values. Model loading may download weights unless local_files_only=True. Long contexts can be truncated: source-level eligibility is not a tokenizer-level guarantee that matching evidence survives.

evidence_plus_answer/evidence_answer_control use notebook 01 exports. Supply variant (occurrence or judge), export_dir, local base_model, initial_adapter (or from_base=True), dev_input and a separately prepared dev_evidence_export in the trainer's v2 schema. A training export is not a dev evidence reference.

rationale_from_base/rationale_continued take accepted rationale_bank, question-level train_source, dev_source, local base_model and initial_adapter. They retain evidence/token-budget checks but no longer require exactly 1,111 questions. replay_fraction controls answer-only replay.

Execution is **off by default**. Preview needs only Python and the repository. Select a kernel with the project's runtime dependencies before executing. Each method uses that kernel's Python in a separate worker process.


In [ ]:
from pathlib import Path
import json
import sys

PROJECT_ROOT = next(
    p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    if (p / "src/notebook_workflows").is_dir()
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from src.notebook_workflows import describe, plan, execute
from src.notebook_workflows.presets import PRESETS, configuration
print("Project:", PROJECT_ROOT)
print("Python:", sys.executable)


## Choose a preset

- **answer_only**: answer_sft
- **evidence_plus_answer**: evidence_sft
- **evidence_answer_control**: evidence_sft
- **rationale_from_base**: rationale_sft
- **rationale_continued**: rationale_sft

Edit the configuration, then rerun the remaining cells. OVERRIDES uses the option names printed by describe(), with underscores. For CLI flags, True includes the named flag and False omits it. Missing inputs block execution.


In [ ]:
CATEGORY = "sft"
PRESET = "answer_only"

DATA = (
    PROJECT_ROOT
    / "data/BioASQ_factoid_sft_prepared"
    / "split_first_train90_dev10_supported_train_seed3407"
)

OVERRIDES = {
    "train_input": [str(DATA / "train_prepared.json")],
    "eval_input": [str(DATA / "dev_prepared.json")],
}

VARIANTS = [
    {
        "label": "qwen05b",
        "overrides": {"model_name": "Qwen/Qwen2.5-0.5B-Instruct"},
    },
    {
        "label": "qwen3b",
        "overrides": {"model_name": "Qwen/Qwen2.5-3B-Instruct"},
    },
]

RUN = False
ALLOW_API = False

METHOD, PARAMETERS = configuration(CATEGORY, PRESET, OVERRIDES)
print(json.dumps({"method": METHOD, "parameters": PARAMETERS}, indent=2))

## Inspect and preview

Options are read from source without importing the training stack. Unknown options and invalid choices are rejected. Outputs go under Artifacts/notebook_runs and cannot overwrite earlier runs. Rerun this cell to allocate fresh plans.


In [ ]:
print(json.dumps(describe(METHOD), indent=2, default=str))
PLANS = []
for variant in VARIANTS:
    method, parameters = configuration(CATEGORY, PRESET, {**OVERRIDES, **variant["overrides"]})
    preview = plan(method, parameters, project_root=PROJECT_ROOT)
    PLANS.append(preview)
    print("\nVariant:", variant["label"])
    print("Method:", preview["method"])
    print("Output:", preview["run_dir"])
    print("Requires GPU:", preview["gpu"], "| Calls API:", preview["api"])
    print("Unresolved inputs:", preview["missing"] or "none")
    print(json.dumps(preview["parameters"], indent=2))


## Execute reviewed plans

Set RUN=True, then rerun configuration and preview. API methods also need ALLOW_API=True. Plans cannot be changed after preview. Each run retains its configuration, input hashes, status and log. A repeated execution of the same plan refuses to overwrite its directory; create a fresh plan instead.


In [ ]:
RESULTS = []
for preview in PLANS:
    if RUN:
        result = execute(preview, run=True, allow_api=ALLOW_API)
        RESULTS.append(str(result))
    else:
        print("Preview only:", preview["method"], "- set RUN=True to execute.")


## Inspect outputs

Use these explicit paths in the next notebook. Historical commands, plots and outputs are preserved in reproducibility/notebook_archive.zip; notebooks/MIGRATION.md maps old notebooks to these workflows.


In [ ]:
for run in RESULTS:
    path = Path(run)
    print("\nRun:", path)
    print((path / "status.json").read_text(encoding="utf-8"))
    print("Log:", path / "run.log")
    print("Outputs:", [p.name for p in path.iterdir()
                      if p.name not in {"run.log", "plan.json", "input_hashes.json"}])


In [ ]:
from IPython.display import Image, display

for run in RESULTS:
    plot = Path(run) / "adapter" / "training_curves.png"
    print(run)
    if plot.exists():
        display(Image(filename=str(plot)))